Cài đặt các thư viện Cần thiêt

In [ ]:
!pip install python-chess
!apt-get update -qq
!apt-get install -y stockfish
!ls /usr/games
!pip install chess tqdm -q

#Phần 1: Train trên tập dữ liệu GM

##Xử lí dữ liệu và load chunk và config


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import chess
import chess.pgn
from tqdm import tqdm


# ══════════════════════════════════════════════════════════
# ❶  CONFIG
# ══════════════════════════════════════════════════════════

PHASE = "pretrain"

PGN_FILES = [
    "/content/drive/MyDrive/datachess/lichess_elite_2024-07.pgn",
    "/content/drive/MyDrive/datachess/lichess_elite_2025-03.pgn",
    "/content/drive/MyDrive/datachess/lichess_elite_2025-09.pgn",
]

DRIVE_DIR      = "/content/drive/MyDrive/datachess"
PRETRAIN_CKPT  = f"{DRIVE_DIR}/pretrain.pt"
EPOCH_CKPT_DIR = f"{DRIVE_DIR}/checkpoints"

CHANNELS        = 64
RES_BLOCKS      = 6
ATTN_HEADS      = 4
ATTN_EVERY      = 3
BATCH_SIZE      = 512          # ↑ tăng được nhờ Mixed Precision tiết kiệm VRAM
LR_PRETRAIN     = 1e-3
EPOCHS_PRETRAIN = 7
MAX_PGN_GAMES   = 250_000
WARMUP_STEPS    = 1000         # Linear warmup trong ~1000 bước đầu
VALUE_WEIGHT    = 1.0          # Trọng số loss của Value Head
LABEL_SMOOTHING = 0.0          # Label Smoothing cho Policy Head
RESUME_EPOCH    = 7

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")   # Mixed Precision chỉ dùng trên GPU
os.makedirs(EPOCH_CKPT_DIR, exist_ok=True)
print(f"✓ Device   : {DEVICE}")
print(f"✓ Phase    : {PHASE}")
print(f"✓ AMP      : {USE_AMP}")
print(f"✓ CKPT dir : {EPOCH_CKPT_DIR}")


# ══════════════════════════════════════════════════════════
# ❷  BOARD ENCODING — CANONICAL
# ══════════════════════════════════════════════════════════

PIECE_MAP = {
    (chess.PAWN,   chess.WHITE): 0,
    (chess.KNIGHT, chess.WHITE): 1,
    (chess.BISHOP, chess.WHITE): 2,
    (chess.ROOK,   chess.WHITE): 3,
    (chess.QUEEN,  chess.WHITE): 4,
    (chess.KING,   chess.WHITE): 5,
    (chess.PAWN,   chess.BLACK): 6,
    (chess.KNIGHT, chess.BLACK): 7,
    (chess.BISHOP, chess.BLACK): 8,
    (chess.ROOK,   chess.BLACK): 9,
    (chess.QUEEN,  chess.BLACK): 10,
    (chess.KING,   chess.BLACK): 11,
}

def board_to_tensor(board: chess.Board, last_move=None) -> np.ndarray:
    tensor = np.zeros((15, 8, 8), dtype=np.float32)
    flip   = (board.turn == chess.BLACK)

    for sq, piece in board.piece_map().items():
        eff_sq = chess.square_mirror(sq) if flip else sq
        row    = eff_sq // 8
        col    = eff_sq % 8
        color  = (not piece.color) if flip else piece.color
        ch     = PIECE_MAP[(piece.piece_type, color)]
        tensor[ch][row][col] = 1.0

    tensor[12] = 1.0  # luôn = 1 sau canonical

    if last_move is not None:
        fr = chess.square_mirror(last_move.from_square) if flip else last_move.from_square
        to = chess.square_mirror(last_move.to_square)   if flip else last_move.to_square
        tensor[13][fr // 8][fr % 8] = 1.0
        tensor[14][to // 8][to % 8] = 1.0

    return tensor


def encode_move_canonical(move: chess.Move, flip: bool) -> int:
    """Encode move với canonical flip."""
    if not flip:
        return move.from_square * 64 + move.to_square
    fr = chess.square_mirror(move.from_square)
    to = chess.square_mirror(move.to_square)
    return fr * 64 + to


def batch_legal_masks(fens: list, device) -> torch.Tensor:
    """Tạo batch masks (B, 4096) từ list FEN strings."""
    masks = torch.full((len(fens), 4096), -1e9, device=device)
    for i, fen in enumerate(fens):
        board = chess.Board(fen)
        flip  = (board.turn == chess.BLACK)
        for move in board.legal_moves:
            masks[i, encode_move_canonical(move, flip)] = 0.0
    return masks


# ══════════════════════════════════════════════════════════
# ❸  WDL LABEL HELPER
# ══════════════════════════════════════════════════════════

# result ∈ {1.0 (White win), 0.0 (Draw), -1.0 (Black win)}
# từ góc nhìn người đang đi: val = result nếu White, -result nếu Black
# val ∈ { 1.0 → Win, 0.0 → Draw, -1.0 → Loss }
RESULT_TO_CLASS = {1.0: 0, 0.0: 1, -1.0: 2}   # Win=0, Draw=1, Loss=2

def val_to_wdl_class(val: float) -> int:
    """Chuyển giá trị vô hướng → index class WDL."""
    return RESULT_TO_CLASS[val]


# ══════════════════════════════════════════════════════════
# ❹  PARSE PGN → CHUNKS  (pipeline mới, không dùng skip_ratio)
# ══════════════════════════════════════════════════════════

RESULT_MAP = {"1-0": 1.0, "0-1": -1.0, "1/2-1/2": 0.0}

def parse_pgn_to_chunks(pgn_paths, chunk_dir,
                        max_games=250_000,
                        chunk_size=30_000):
    """
    Pipeline lọc dữ liệu:
      1. Bỏ ván < 40 plies (< 20 moves)
      2. Bỏ 8-10 plies đầu  (random mỗi ván — tránh khai cuộc cứng)
      3. Bỏ 8-10 plies cuối (random mỗi ván — tránh blunder tàn cục)
      4. Lưu 1 position mỗi 2 plies với random offset {0,1}
         → cân bằng White/Black sau Canonical Flip
    """
    os.makedirs(chunk_dir, exist_ok=True)

    existing = sorted([
        os.path.join(chunk_dir, f)
        for f in os.listdir(chunk_dir) if f.endswith(".npz")
    ])
    if existing:
        print(f"✓ Tìm thấy {len(existing)} chunks: {chunk_dir}")
        return existing

    # lưu cả wdl_classes để train Value Head kiểu WDL
    states, moves, values, wdl_classes, fens = [], [], [], [], []
    total_games = 0
    chunk_idx   = 0
    chunk_files = []

    def flush_chunk():
        nonlocal chunk_idx
        if not states:
            return
        chunk_path = os.path.join(chunk_dir, f"chunk_{chunk_idx:04d}.npz")
        np.savez_compressed(
            chunk_path,
            states     = np.array(states,      dtype=np.float32),
            moves      = np.array(moves,       dtype=np.int64),
            evals      = np.array(values,      dtype=np.float32),
            wdl        = np.array(wdl_classes, dtype=np.int64),
            fens       = np.array(fens,        dtype=object),
        )
        print(f"  💾 Chunk {chunk_idx:04d}: {len(states):,} positions → {chunk_path}")
        chunk_files.append(chunk_path)
        states.clear(); moves.clear(); values.clear()
        wdl_classes.clear(); fens.clear()
        chunk_idx += 1

    for pgn_path in pgn_paths:
        if not os.path.exists(pgn_path):
            print(f"⚠ Không tìm thấy: {pgn_path}, bỏ qua.")
            continue

        print(f"\n📂 Parsing: {pgn_path}")
        with open(pgn_path, encoding="utf-8", errors="ignore") as f:
            while total_games < max_games:
                game = chess.pgn.read_game(f)
                if game is None:
                    break

                result = RESULT_MAP.get(game.headers.get("Result"), None)
                if result is None:
                    continue

                game_moves  = list(game.mainline_moves())
                total_plies = len(game_moves)

                # ❶ Bỏ ván quá ngắn
                if total_plies < 40:
                    continue

                # ❷ Random số plies bỏ đầu & cuối
                skip_start = random.randint(8, 10)
                skip_end   = random.randint(8, 10)
                end_ply    = total_plies - skip_end

                # ❸ Random offset — cân bằng White/Black
                offset = random.choice([0, 1])

                board     = game.board()
                last_move = None

                for i, move in enumerate(game_moves):
                    if i < skip_start:
                        last_move = move
                        board.push(move)
                        continue

                    if i >= end_ply:
                        break

                    relative = i - skip_start
                    if (relative - offset) % 2 == 0:
                        flip = (board.turn == chess.BLACK)
                        val  = result if board.turn == chess.WHITE else -result

                        states.append(board_to_tensor(board, last_move))
                        moves.append(encode_move_canonical(move, flip))
                        values.append(val)
                        wdl_classes.append(val_to_wdl_class(val))
                        fens.append(board.fen())

                    last_move = move
                    board.push(move)

                total_games += 1
                if total_games % chunk_size == 0:
                    print(f"  {total_games:>6} games parsed...")
                    flush_chunk()

        if total_games >= max_games:
            break

    flush_chunk()
    print(f"\n✓ Tổng: {total_games:,} games | {len(chunk_files)} chunks")
    return chunk_files


# ══════════════════════════════════════════════════════════
# ❺  DATASET + SAMPLER
# ══════════════════════════════════════════════════════════

class MultiChunkDataset(Dataset):
    def __init__(self, chunk_files: list):
        self.chunk_files = chunk_files
        self.chunk_sizes = []

        print("Đang đọc metadata chunks...")
        for cf in chunk_files:
            d = np.load(cf, allow_pickle=True)
            self.chunk_sizes.append(len(d["moves"]))

        self.cumulative    = np.cumsum([0] + self.chunk_sizes)
        self._cached_idx   = -1
        self._cached_chunk = None

        total = self.cumulative[-1]
        print(f"  → {total:,} positions từ {len(chunk_files)} chunks")
        print(f"  → RAM peak: ~{max(self.chunk_sizes) * 3840 / 1e9:.2f} GB (1 chunk)")

    def __len__(self):
        return int(self.cumulative[-1])

    def _load_chunk(self, chunk_idx: int):
        if self._cached_idx != chunk_idx:
            d = np.load(self.chunk_files[chunk_idx], allow_pickle=True)
            self._cached_chunk = {
                "states": torch.from_numpy(d["states"]),
                "moves" : torch.from_numpy(d["moves"]),
                "evals" : torch.from_numpy(d["evals"]),
                "wdl"   : torch.from_numpy(d["wdl"]),
                "fens"  : d["fens"],
            }
            self._cached_idx = chunk_idx

    def __getitem__(self, idx):
        chunk_idx = int(np.searchsorted(self.cumulative[1:], idx, side="right"))
        local_idx = idx - int(self.cumulative[chunk_idx])
        self._load_chunk(chunk_idx)
        c = self._cached_chunk
        return (
            c["states"][local_idx],
            c["moves"][local_idx],
            c["evals"][local_idx],
            c["wdl"][local_idx],
            str(c["fens"][local_idx]),
        )


class ChunkSampler(torch.utils.data.Sampler):
    def __init__(self, dataset: MultiChunkDataset, shuffle: bool = True):
        self.dataset  = dataset
        self.shuffle  = shuffle
        self.n_chunks = len(dataset.chunk_files)

    def __len__(self):
        return len(self.dataset)

    def __iter__(self):
        chunk_order = list(range(self.n_chunks))
        if self.shuffle:
            np.random.shuffle(chunk_order)
        for ci in chunk_order:
            start   = int(self.dataset.cumulative[ci])
            end     = int(self.dataset.cumulative[ci + 1])
            indices = np.arange(start, end)
            if self.shuffle:
                np.random.shuffle(indices)
            yield from indices.tolist()


def collate_with_fen(batch):
    """Custom collate — giữ FEN là list string, không tensor hóa."""
    states = torch.stack([b[0] for b in batch])
    moves  = torch.stack([b[1] for b in batch])
    evals  = torch.stack([b[2] for b in batch])
    wdl    = torch.stack([b[3] for b in batch])
    fens   = [b[4] for b in batch]
    return states, moves, evals, wdl, fens


def make_chunk_loaders(dataset: MultiChunkDataset, val_ratio=0.1, batch_size=512):
    n_val   = max(1, int(len(dataset.chunk_files) * val_ratio))
    n_train = len(dataset.chunk_files) - n_val

    train_ds = MultiChunkDataset(dataset.chunk_files[:n_train])
    val_ds   = MultiChunkDataset(dataset.chunk_files[n_train:])

    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        sampler=ChunkSampler(train_ds, shuffle=True),
        num_workers=0, pin_memory=True,
        collate_fn=collate_with_fen,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size,
        sampler=ChunkSampler(val_ds, shuffle=False),
        num_workers=0, pin_memory=True,
        collate_fn=collate_with_fen,
    )
    print(f"Train: {len(train_ds):,} ({n_train} chunks) | "
          f"Val: {len(val_ds):,} ({n_val} chunks)")
    return train_loader, val_loader


##Kiến trúc model và checckpoint

In [ ]:
# ══════════════════════════════════════════════════════════
# ❻  MODEL  (Value Head: 1 nơ-ron → 3 nơ-ron cho WDL)
# ══════════════════════════════════════════════════════════

class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.se    = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(channels, channels // 4), nn.ReLU(),
            nn.Linear(channels // 4, channels), nn.Sigmoid(),
        )

    def forward(self, x):
        r = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = x * self.se(x).view(x.size(0), -1, 1, 1)
        return F.relu(x + r)


class BoardAttention(nn.Module):
    def __init__(self, channels: int, heads: int = 4):
        super().__init__()
        self.attn  = nn.MultiheadAttention(channels, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(channels)
        self.ff    = nn.Sequential(
            nn.Linear(channels, channels * 2),
            nn.ReLU(),
            nn.Linear(channels * 2, channels),
        )
        self.norm2 = nn.LayerNorm(channels)

    def forward(self, x):
        B, C, H, W = x.shape
        s = x.flatten(2).permute(0, 2, 1)
        a, _ = self.attn(s, s, s)
        s = self.norm1(s + a)
        s = self.norm2(s + self.ff(s))
        return s.permute(0, 2, 1).view(B, C, H, W)


class ChessNet(nn.Module):
    def __init__(self, in_channels=15, channels=64,
                 res_blocks=6, attn_heads=4, attn_every=3):
        super().__init__()
        self.input_conv = nn.Sequential(
            nn.Conv2d(in_channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        )
        layers = []
        for i in range(res_blocks):
            layers.append(ResBlock(channels))
            if (i + 1) % attn_every == 0:
                layers.append(BoardAttention(channels, attn_heads))
        self.backbone = nn.Sequential(*layers)

        # Policy Head (không đổi)
        self.policy_conv = nn.Conv2d(channels, 32, 1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(32)
        self.policy_fc   = nn.Linear(32 * 8 * 8, 4096)

        # Value Head: 1 nơ-ron tanh → 3 nơ-ron WDL (logits, không dùng tanh)
        self.value_conv = nn.Conv2d(channels, 8, 1, bias=False)
        self.value_bn   = nn.BatchNorm2d(8)
        self.value_fc1  = nn.Linear(8 * 8 * 8, 64)
        self.value_fc2  = nn.Linear(64, 3)   # ← WDL: Win / Draw / Loss

    def forward(self, x):
        x = self.input_conv(x)
        x = self.backbone(x)

        p = F.relu(self.policy_bn(self.policy_conv(x))).flatten(1)
        p = self.policy_fc(p)                          # logits (4096,)

        v = F.relu(self.value_bn(self.value_conv(x))).flatten(1)
        v = F.relu(self.value_fc1(v))
        v = self.value_fc2(v)                          # logits WDL (3,), không tanh

        return p, v

    def value_scalar(self, x):
        """Trả về scalar ∈ [-1,1] để dùng khi eval/chơi: E[V] = P(Win) - P(Loss)."""
        _, v_logits = self.forward(x)
        probs = torch.softmax(v_logits, dim=-1)        # [B, 3]
        return probs[:, 0] - probs[:, 2]               # P(Win) - P(Loss)


# ══════════════════════════════════════════════════════════
# ❼  WARMUP + COSINE SCHEDULER
# ══════════════════════════════════════════════════════════

def make_scheduler(optimizer, warmup_steps: int, total_steps: int, eta_min: float = 1e-5):
    """
    Linear warmup trong warmup_steps bước đầu,
    sau đó Cosine Annealing cho phần còn lại.
    """
    def warmup_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)   # tăng tuyến tính từ 0 → 1
        return 1.0

    warmup_sched  = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_lambda)
    cosine_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps - warmup_steps, eta_min=eta_min
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[warmup_steps],
    )


# ══════════════════════════════════════════════════════════
# ❽  CHECKPOINT HELPERS
# ══════════════════════════════════════════════════════════

def save_checkpoint(model, optimizer, scheduler, scaler, epoch, best_acc, phase, ckpt_dir):
    path = os.path.join(ckpt_dir, f"{phase}_epoch{epoch:02d}.pt")
    torch.save({
        "epoch"     : epoch,
        "model"     : model.state_dict(),
        "optimizer" : optimizer.state_dict(),
        "scheduler" : scheduler.state_dict(),
        "scaler"    : scaler.state_dict() if scaler else None,
        "best_acc"  : best_acc,
    }, path)
    print(f"  💾 Checkpoint → {path}")
    return path


def load_checkpoint(model, optimizer, scheduler, scaler, ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    if scaler and ckpt.get("scaler"):
        scaler.load_state_dict(ckpt["scaler"])
    print(f"✓ Resumed từ: {ckpt_path} (epoch {ckpt['epoch']})")
    return ckpt["epoch"], ckpt["best_acc"]


# ══════════════════════════════════════════════════════════
# ❾  TRAINING LOOP
# ══════════════════════════════════════════════════════════

def pretrain_epoch(model, loader, optimizer, scheduler, scaler, device,
                   value_weight=1.0, label_smoothing=0.1):
    """
    Cải tiến so với v2:
      • Mixed Precision (autocast + GradScaler)
      • Label Smoothing cho Policy Loss
      • WDL CrossEntropy cho Value Loss
      • Gradient Clipping (max_norm=1.0)
      • Per-step LR scheduling (warmup + cosine)
    """
    model.train()
    total_loss = total_correct = total = 0

    pbar = tqdm(loader, desc="  Pretrain", leave=False, dynamic_ncols=True)
    for states, moves, _, wdl, fens in pbar:
        states = states.to(device)
        moves  = moves.to(device)
        wdl    = wdl.to(device)

        optimizer.zero_grad()

        # ── Forward + Loss dưới autocast ──────────────────
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            policy_out, value_out = model(states)

            masks         = batch_legal_masks(fens, device)
            masked_logits = policy_out + masks

            # Policy loss với Label Smoothing
            policy_loss = F.cross_entropy(
                masked_logits, moves, label_smoothing=label_smoothing
            )
            # Value loss: WDL CrossEntropy
            value_loss = F.cross_entropy(value_out, wdl)

            loss = policy_loss + value_weight * value_loss

        # ── Backward với GradScaler ───────────────────────
        scaler.scale(loss).backward()

        # Gradient Clipping (unscale trước khi clip)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        # Per-step scheduler step
        scheduler.step()

        batch_correct  = (masked_logits.argmax(1) == moves).sum().item()
        total_loss    += loss.item() * len(states)
        total_correct += batch_correct
        total         += len(states)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            p_loss=f"{policy_loss.item():.4f}",
            v_loss=f"{value_loss.item():.4f}",
            acc=f"{batch_correct/len(states):.3f}",
        )

    return total_loss / total, total_correct / total


def evaluate(model, loader, device):
    model.eval()
    total_correct = total_top5 = total = 0

    with torch.no_grad():
        for states, moves, _, wdl, fens in tqdm(loader, desc="  Eval",
                                                 leave=False, dynamic_ncols=True):
            states = states.to(device)
            moves  = moves.to(device)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                policy_out, _ = model(states)
                masks         = batch_legal_masks(fens, device)
                masked_logits = policy_out + masks

            total_correct += (masked_logits.argmax(1) == moves).sum().item()
            top5 = masked_logits.topk(5, dim=1).indices
            total_top5 += (top5 == moves.unsqueeze(1)).any(1).sum().item()
            total += len(states)

    return total_correct / total, total_top5 / total

##Giai đoạn training model


In [ ]:
# ══════════════════════════════════════════════════════════
# ❿  MAIN
# ══════════════════════════════════════════════════════════

def run_train():
    print("=" * 60)
    print("  PRETRAIN v3  (WDL · Label Smoothing · AMP · Warmup)")
    print("=" * 60)

    CHUNK_DIR = f"{DRIVE_DIR}/pgn_cache_chunks_v3"
    chunk_files = parse_pgn_to_chunks(
        PGN_FILES, CHUNK_DIR,
        max_games=MAX_PGN_GAMES,
    )

    dataset = MultiChunkDataset(chunk_files)
    train_loader, val_loader = make_chunk_loaders(
        dataset, val_ratio=0.1, batch_size=BATCH_SIZE
    )

    model = ChessNet(channels=CHANNELS, res_blocks=RES_BLOCKS,
                     attn_heads=ATTN_HEADS, attn_every=ATTN_EVERY).to(DEVICE)
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

    # AdamW thay vì Adam — weight decay chuẩn cho Transformer
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PRETRAIN,
                                  weight_decay=1e-4)

    # Tính tổng số bước để tạo scheduler
    steps_per_epoch = len(train_loader)
    total_steps     = steps_per_epoch * EPOCHS_PRETRAIN
    scheduler = make_scheduler(optimizer,
                               warmup_steps=WARMUP_STEPS,
                               total_steps=total_steps,
                               eta_min=1e-5)

    # GradScaler cho Mixed Precision (no-op trên CPU)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    start_epoch = 1
    best_acc    = 0.0

    if RESUME_EPOCH > 0:
        ckpt_path = os.path.join(EPOCH_CKPT_DIR,
                                 f"pretrain_epoch{RESUME_EPOCH:02d}.pt")
        if os.path.exists(ckpt_path):
            start_epoch, best_acc = load_checkpoint(
                model, optimizer, scheduler, scaler, ckpt_path, DEVICE)
            start_epoch += 1
        else:
            print(f"⚠ Không tìm thấy checkpoint epoch {RESUME_EPOCH}, train từ đầu.")

    print(f"\nBắt đầu từ epoch {start_epoch}/{EPOCHS_PRETRAIN}")
    for epoch in range(start_epoch, EPOCHS_PRETRAIN + 1):
        current_lr = optimizer.param_groups[0]["lr"]
        print(f"\nEpoch {epoch:02d}/{EPOCHS_PRETRAIN} | LR: {current_lr:.2e}")

        train_loss, train_acc = pretrain_epoch(
            model, train_loader, optimizer, scheduler, scaler, DEVICE,
            value_weight=VALUE_WEIGHT, label_smoothing=LABEL_SMOOTHING
        )
        val_acc, val_top5 = evaluate(model, val_loader, DEVICE)
        # Lưu ý: scheduler đã được step() mỗi batch trong pretrain_epoch,
        # nên KHÔNG gọi scheduler.step() thêm ở đây.

        print(f"  Loss: {train_loss:.4f} | Train acc: {train_acc:.3f} | "
              f"Val Top-1: {val_acc:.3f} | Top-5: {val_top5:.3f}")

        save_checkpoint(model, optimizer, scheduler, scaler,
                        epoch, best_acc, "pretrain", EPOCH_CKPT_DIR)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), PRETRAIN_CKPT)
            print(f"  ✓ Best saved → {PRETRAIN_CKPT} (acc: {best_acc:.4f})")

    print(f"\n✅ Pretrain xong! Best val acc: {best_acc:.4f}")


run_train()

#Phần 2: giai đoạn học tăng cường

In [ ]:
import copy
import math
import random
import threading
import chess
import chess.engine
from collections import deque
from typing import List, Optional

# ══════════════════════════════════════════════════════════
# CONFIG RL
# ══════════════════════════════════════════════════════════

RL_DIR               = f"{DRIVE_DIR}/rl"
BEST_MODEL_CKPT      = f"{RL_DIR}/best_model.pt"
RL_CKPT_DIR          = f"{RL_DIR}/checkpoints"

NUM_SELF_PLAY_GAMES  = 128    # chia đôi cho 2 GPU: 32 ván/GPU
MCTS_SIMS_SELFPLAY   = 400
MCTS_SIMS_EVAL       = 400
MAX_GAME_MOVES       = 150
TEMP_THRESHOLD       = 30

C_PUCT               = 2.0   # sweet spot từ sweep

RL_BATCH_SIZE        = 512
RL_LR                = 1e-4
RL_EPOCHS            = 3
RL_WEIGHT_DECAY      = 1e-4

EVAL_GAMES           = 40
WIN_THRESHOLD        = 0.55
RL_ITERATIONS        = 25

# Detect GPUs
N_GPUS = torch.cuda.device_count()
DEV0   = torch.device("cuda:0")
DEV1   = torch.device("cuda:1") if N_GPUS >= 2 else DEV0

os.makedirs(RL_DIR,      exist_ok=True)
os.makedirs(RL_CKPT_DIR, exist_ok=True)
print(f"✓ RL dir   : {RL_DIR}")
print(f"✓ AMP      : {USE_AMP}")
print(f"✓ c_puct   : {C_PUCT}")
print(f"✓ GPUs     : {N_GPUS} detected → DEV0={DEV0}, DEV1={DEV1}")


# ══════════════════════════════════════════════════════════
# ❶  MCTS NODE
#    Convention: node.avg_value = Q từ góc nhìn PARENT
# ══════════════════════════════════════════════════════════

class MCTSNode:
    __slots__ = ["parent", "move", "prior_p",
                 "children", "visit_count", "total_value", "avg_value"]

    def __init__(self, parent=None, move=None, prior_p=0.0):
        self.parent      = parent
        self.move        = move
        self.prior_p     = prior_p
        self.children    = {}
        self.visit_count = 0
        self.total_value = 0.0
        self.avg_value   = 0.0

    def is_leaf(self):
        return len(self.children) == 0

    def get_puct(self, c_puct):
        u = (c_puct * self.prior_p
             * math.sqrt(self.parent.visit_count)
             / (1 + self.visit_count))
        return self.avg_value + u


# ══════════════════════════════════════════════════════════
# ❷  BATCHED MCTS ENGINE
# ══════════════════════════════════════════════════════════

class BatchedMCTS:
    def __init__(self, model, device,
                 n_games: int = 32,
                 c_puct: float = C_PUCT,
                 dirichlet_alpha: float = 0.3,
                 dirichlet_eps: float = 0.25):
        self.model           = model
        self.device          = device
        self.n_games         = n_games
        self.c_puct          = c_puct
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_eps   = dirichlet_eps

        self.roots      : List[Optional[MCTSNode]]    = [None] * n_games
        self.boards     : List[Optional[chess.Board]] = [None] * n_games
        self.last_moves : List[Optional[chess.Move]]  = [None] * n_games
        self.active     : List[bool]                  = [False] * n_games

    def init_games(self, boards, last_moves, add_noise=True):
        assert len(boards) == self.n_games
        tensors = []
        for i, (board, lm) in enumerate(zip(boards, last_moves)):
            self.boards[i]     = board
            self.last_moves[i] = lm
            self.roots[i]      = MCTSNode()
            self.active[i]     = True
            tensors.append(board_to_tensor(board, lm))

        priors_list, _ = self._batch_forward(tensors, boards)

        for i in range(self.n_games):
            if not self.active[i]:
                continue
            if boards[i].is_game_over():
                self.active[i] = False
                continue
            self._do_expand(self.roots[i], boards[i], priors_list[i])
            self.roots[i].visit_count = 1
            if add_noise:
                self._add_dirichlet(self.roots[i])

    def step(self):
        """Selection → Batch forward → Expand → Backprop.
        Dùng push/pop thay board.copy() để tránh deep-copy."""
        leaf_nodes   = [None] * self.n_games
        leaf_lmoves  = [None] * self.n_games
        leaf_tensors = []
        active_idx   = []   # (game_idx, moves_made)

        for i in range(self.n_games):
            if not self.active[i] or self.roots[i] is None:
                leaf_tensors.append(None)
                continue

            node       = self.roots[i]
            board      = self.boards[i]
            last_sim   = self.last_moves[i]
            moves_made = []

            while not node.is_leaf():
                best_move = max(
                    node.children,
                    key=lambda m: node.children[m].get_puct(self.c_puct)
                )
                node = node.children[best_move]
                board.push(best_move)
                moves_made.append(best_move)
                last_sim = best_move

            leaf_nodes[i]  = node
            leaf_lmoves[i] = last_sim

            if board.is_game_over():
                leaf_tensors.append(None)
                outcome = board.outcome()
                value   = 0.0 if (outcome is None or outcome.winner is None) else -1.0
                for _ in moves_made:
                    board.pop()
                self._backprop(node, value)
            else:
                leaf_tensors.append(board_to_tensor(board, last_sim))
                active_idx.append((i, moves_made))

        if not active_idx:
            return

        active_i     = [x[0] for x in active_idx]
        active_moves = [x[1] for x in active_idx]

        tensors_batch = [leaf_tensors[i] for i in active_i]
        boards_batch  = [self.boards[i]  for i in active_i]
        priors_list, values = self._batch_forward(tensors_batch, boards_batch)

        for j, (i, moves_made) in enumerate(zip(active_i, active_moves)):
            self._do_expand(leaf_nodes[i], self.boards[i], priors_list[j])
            for _ in moves_made:
                self.boards[i].pop()
            self._backprop(leaf_nodes[i], values[j])

    def get_results(self, boards):
        policies   = []
        best_moves = []
        for i in range(self.n_games):
            root  = self.roots[i]
            board = boards[i]
            flip  = (board.turn == chess.BLACK)
            counts = np.zeros(4096, dtype=np.float32)
            if root and root.children:
                for move, child in root.children.items():
                    counts[encode_move_canonical(move, flip)] = child.visit_count
            total = counts.sum()
            policies.append(counts / total if total > 0 else counts)
            if root and root.children:
                best_moves.append(max(root.children,
                                      key=lambda m: root.children[m].visit_count))
            else:
                legal = list(board.legal_moves)
                best_moves.append(legal[0] if legal else None)
        return np.array(policies, dtype=np.float32), best_moves

    def partial_reinit(self, active_indices, add_noise=True):
        if not active_indices:
            return
        tensors       = [board_to_tensor(self.boards[i], self.last_moves[i])
                         for i in active_indices]
        boards_reinit = [self.boards[i] for i in active_indices]
        priors_list, _ = self._batch_forward(tensors, boards_reinit)
        for j, i in enumerate(active_indices):
            board = self.boards[i]
            if board.is_game_over():
                self.active[i] = False
                self.roots[i]  = None
                continue
            self.roots[i]             = MCTSNode()
            self.roots[i].visit_count = 1
            self._do_expand(self.roots[i], board, priors_list[j])
            if add_noise:
                self._add_dirichlet(self.roots[i])
            self.active[i] = True

    @torch.no_grad()
    def _batch_forward(self, tensors, boards):
        batch = torch.from_numpy(np.stack(tensors)).to(self.device)
        n     = len(tensors)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            policy_logits, value_logits = self.model(batch)
            wdl_probs = torch.softmax(value_logits, dim=-1)
            values_t  = wdl_probs[:, 0] - wdl_probs[:, 2]

            # Build mask trên CPU numpy → push GPU 1 lần
            cpu_mask = np.full((n, 4096), -1e4, dtype=np.float32)
            for j, board in enumerate(boards):
                flip = (board.turn == chess.BLACK)
                for m in board.legal_moves:
                    cpu_mask[j, encode_move_canonical(m, flip)] = 0.0
            mask         = torch.from_numpy(cpu_mask).to(policy_logits.device)
            policy_probs = torch.softmax(policy_logits + mask, dim=1).cpu().numpy()

        values = values_t.cpu().numpy().tolist()

        priors_list = []
        for j, board in enumerate(boards):
            flip        = (board.turn == chess.BLACK)
            legal_moves = list(board.legal_moves)
            priors      = {m: float(policy_probs[j, encode_move_canonical(m, flip)])
                           for m in legal_moves}
            p_sum = sum(priors.values())
            if p_sum > 1e-8:
                priors = {m: p / p_sum for m, p in priors.items()}
            else:
                priors = {m: 1.0 / len(legal_moves) for m in legal_moves}
            priors_list.append(priors)

        return priors_list, values

    def _do_expand(self, node, board, priors):
        for move, prior in priors.items():
            node.children[move] = MCTSNode(parent=node, move=move, prior_p=prior)

    def _backprop(self, node, value):
        while node.parent is not None:
            node.visit_count  += 1
            node.total_value  -= value
            node.avg_value     = node.total_value / node.visit_count
            value              = -value
            node               = node.parent
        node.visit_count += 1

    def _add_dirichlet(self, root):
        if not root.children:
            return
        moves = list(root.children.keys())
        noise = np.random.dirichlet([self.dirichlet_alpha] * len(moves))
        for m, n in zip(moves, noise):
            c         = root.children[m]
            c.prior_p = ((1 - self.dirichlet_eps) * c.prior_p
                         + self.dirichlet_eps * n)


# ══════════════════════════════════════════════════════════
# ❸  SINGLE-GAME MCTS (eval + vs Stockfish)
# ══════════════════════════════════════════════════════════

class MCTSEngine:
    def __init__(self, model, device, c_puct=C_PUCT,
                 dirichlet_alpha=0.3, dirichlet_eps=0.25):
        self.model           = model
        self.device          = device
        self.c_puct          = c_puct
        self.dirichlet_alpha = dirichlet_alpha
        self.dirichlet_eps   = dirichlet_eps

    @torch.no_grad()
    def _evaluate_leaf(self, board, last_move):
        flip   = (board.turn == chess.BLACK)
        tensor = torch.from_numpy(
            board_to_tensor(board, last_move)
        ).unsqueeze(0).to(self.device)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            policy_logits, value_logits = self.model(tensor)
            wdl_probs = torch.softmax(value_logits, dim=-1)
            value     = float((wdl_probs[0, 0] - wdl_probs[0, 2]).item())
        policy_probs = torch.softmax(policy_logits, dim=1).squeeze(0).cpu().numpy()
        legal_moves  = list(board.legal_moves)
        priors       = {}
        p_sum        = 0.0
        for m in legal_moves:
            idx       = encode_move_canonical(m, flip)
            p         = max(float(policy_probs[idx]), 0.0)
            priors[m] = p
            p_sum    += p
        if p_sum > 1e-8:
            for m in priors: priors[m] /= p_sum
        else:
            uniform = 1.0 / max(len(legal_moves), 1)
            for m in legal_moves: priors[m] = uniform
        return priors, value

    def _do_expand(self, node, board, priors):
        for move, prior in priors.items():
            node.children[move] = MCTSNode(parent=node, move=move, prior_p=prior)

    def _backprop(self, node, value):
        while node.parent is not None:
            node.visit_count  += 1
            node.total_value  -= value
            node.avg_value     = node.total_value / node.visit_count
            value              = -value
            node               = node.parent
        node.visit_count += 1

    def search(self, board, last_move, num_simulations=200, add_noise=False):
        root = MCTSNode()
        if board.is_game_over():
            return root
        priors, _ = self._evaluate_leaf(board.copy(), last_move)
        self._do_expand(root, board, priors)
        root.visit_count = 1
        if add_noise:
            moves = list(root.children.keys())
            noise = np.random.dirichlet([self.dirichlet_alpha] * len(moves))
            for m, n in zip(moves, noise):
                c = root.children[m]
                c.prior_p = ((1 - self.dirichlet_eps) * c.prior_p
                             + self.dirichlet_eps * n)
        for _ in range(num_simulations):
            node      = root
            board_sim = board.copy()
            last_sim  = last_move
            while not node.is_leaf():
                best_move = max(node.children,
                                key=lambda m: node.children[m].get_puct(self.c_puct))
                node = node.children[best_move]
                board_sim.push(best_move)
                last_sim = best_move
            if board_sim.is_game_over():
                outcome = board_sim.outcome()
                value   = 0.0 if (outcome is None or outcome.winner is None) else -1.0
            else:
                priors, value = self._evaluate_leaf(board_sim, last_sim)
                self._do_expand(node, board_sim, priors)
            self._backprop(node, value)
        return root

    def best_move(self, board, last_move, num_simulations=200):
        root = self.search(board, last_move, num_simulations, add_noise=False)
        if not root.children:
            legal = list(board.legal_moves)
            return legal[0] if legal else None
        return max(root.children,
                   key=lambda m: root.children[m].visit_count)


# ══════════════════════════════════════════════════════════
# ❹  DECODE MOVE
# ══════════════════════════════════════════════════════════

def decode_move_canonical(move_idx, board, flip):
    from_sq = move_idx // 64
    to_sq   = move_idx % 64
    if flip:
        from_sq = chess.square_mirror(from_sq)
        to_sq   = chess.square_mirror(to_sq)
    move = chess.Move(from_sq, to_sq)
    if move in board.legal_moves:
        return move
    for promo in [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]:
        pm = chess.Move(from_sq, to_sq, promotion=promo)
        if pm in board.legal_moves:
            return pm
    return None


# ══════════════════════════════════════════════════════════
# ❺  OPENING BOOK (cho eval)
# ══════════════════════════════════════════════════════════

_OPENING_LINES = [
    ["e4", "e5"],                               ["e4", "c5"],
    ["e4", "e6"],                               ["e4", "c6"],
    ["e4", "d5"],                               ["d4", "d5"],
    ["d4", "Nf6"],                              ["d4", "f5"],
    ["c4", "e5"],                               ["c4", "Nf6"],
    ["Nf3", "d5"],                              ["Nf3", "Nf6"],
    ["e4", "e5", "Nf3", "Nc6"],                 ["e4", "e5", "Nf3", "Nf6"],
    ["e4", "e5", "Nf3", "Nc6", "Bc4"],
    ["d4", "d5", "c4", "e6"],                   ["d4", "d5", "c4", "c6"],
    ["d4", "Nf6", "c4", "e6"],                  ["d4", "Nf6", "c4", "g6"],
    ["e4", "c5", "Nf3", "d6"],
]

def _random_opening(board):
    line      = random.choice(_OPENING_LINES)
    last_move = None
    for san in line:
        try:
            move = board.parse_san(san)
            if move in board.legal_moves:
                board.push(move)
                last_move = move
            else:
                break
        except Exception:
            break
    return last_move


# ══════════════════════════════════════════════════════════
# ❻  SINGLE-GPU SELF-PLAY (core logic, tái dùng cho 2 GPU)
# ══════════════════════════════════════════════════════════

def _selfplay_on_device(model, device, n_games,
                         num_simulations, max_moves, temp_threshold,
                         out_list, idx):
    """
    Chạy n_games ván trên device cụ thể.
    Kết quả ghi vào out_list[idx] = (states, policies, wdl, fens).
    Model đã được đưa lên đúng GPU bởi caller trước khi truyền vào.
    """
    model.eval()

    boards      = [chess.Board() for _ in range(n_games)]
    last_moves  = [None] * n_games
    move_counts = [0]    * n_games
    done        = [False] * n_games

    game_states   = [[] for _ in range(n_games)]
    game_policies = [[] for _ in range(n_games)]
    game_turns    = [[] for _ in range(n_games)]
    game_fens     = [[] for _ in range(n_games)]
    results       = {"1-0": 0, "0-1": 0, "1/2-1/2": 0, "*": 0}

    engine = BatchedMCTS(model, device, n_games=n_games,
                         c_puct=C_PUCT,
                         dirichlet_alpha=0.3, dirichlet_eps=0.25)
    engine.init_games(boards, last_moves, add_noise=True)

    while not all(done):
        active_idx = [i for i in range(n_games) if not done[i]]
        if not active_idx:
            break

        for _ in range(num_simulations):
            engine.step()

        policies, best_moves = engine.get_results(boards)
        newly_done = []

        for i in active_idx:
            board   = boards[i]
            policy  = policies[i]
            best_mv = best_moves[i]

            if policy.sum() <= 0 or best_mv is None:
                done[i] = True
                newly_done.append(i)
                continue

            flip = (board.turn == chess.BLACK)
            if move_counts[i] < temp_threshold:
                root   = engine.roots[i]
                counts = np.zeros(4096, dtype=np.float32)
                for move, child in root.children.items():
                    counts[encode_move_canonical(move, flip)] = child.visit_count
                valid_idx = np.where(counts > 0)[0]
                if len(valid_idx) > 0:
                    valid_prob = counts[valid_idx] / counts[valid_idx].sum()
                    move_idx   = np.random.choice(valid_idx, p=valid_prob)
                    chosen     = decode_move_canonical(int(move_idx), board, flip)
                    if chosen is None or chosen not in board.legal_moves:
                        chosen = best_mv
                else:
                    chosen = best_mv
            else:
                chosen = best_mv

            if chosen not in board.legal_moves:
                legal  = list(board.legal_moves)
                chosen = legal[0] if legal else None
            if chosen is None:
                done[i] = True
                newly_done.append(i)
                continue

            game_states[i].append(board_to_tensor(board, last_moves[i]))
            game_policies[i].append(policy)
            game_turns[i].append(board.turn)
            game_fens[i].append(board.fen())

            board.push(chosen)
            last_moves[i]        = chosen
            engine.last_moves[i] = chosen
            move_counts[i]      += 1

            if board.is_game_over() or move_counts[i] >= max_moves:
                result = board.result()
                results[result] = results.get(result, 0) + 1
                done[i] = True
                newly_done.append(i)

        for i in newly_done:
            engine.active[i] = False
            engine.roots[i]  = None

        still_active = [i for i in active_idx if not done[i]]
        if still_active:
            engine.partial_reinit(still_active, add_noise=True)

    # Gom data
    all_states, all_policies, all_wdl, all_fens = [], [], [], []
    for i in range(n_games):
        if not game_states[i]:
            continue
        result_str = boards[i].result()
        for state, policy, turn, fen in zip(
                game_states[i], game_policies[i],
                game_turns[i], game_fens[i]):
            if result_str == "1/2-1/2":
                wdl = 1
            elif (result_str == "1-0" and turn == chess.WHITE) or \
                 (result_str == "0-1" and turn == chess.BLACK):
                wdl = 0
            else:
                wdl = 2
            all_states.append(state)
            all_policies.append(policy)
            all_wdl.append(wdl)
            all_fens.append(fen)

    if not all_states:
        out_list[idx] = (np.zeros((0,15,8,8), dtype=np.float32),
                         np.zeros((0,4096),   dtype=np.float32),
                         np.zeros((0,),       dtype=np.int64),
                         [])
        return

    states_cat   = np.array(all_states,   dtype=np.float32)
    policies_cat = np.array(all_policies, dtype=np.float32)
    wdl_cat      = np.array(all_wdl,      dtype=np.int64)

    valid_mask   = policies_cat.sum(axis=1) > 0
    states_cat   = states_cat[valid_mask]
    policies_cat = policies_cat[valid_mask]
    wdl_cat      = wdl_cat[valid_mask]
    all_fens     = [f for f, v in zip(all_fens, valid_mask) if v]

    print(f"  [GPU {device}] {len(states_cat):,} pos | {results}")
    out_list[idx] = (states_cat, policies_cat, wdl_cat, all_fens)


# ══════════════════════════════════════════════════════════
# ❼  DUAL-GPU SELF-PLAY  (multiprocessing, không bị GIL)
# ══════════════════════════════════════════════════════════

def generate_self_play_2gpu(model, n_games=NUM_SELF_PLAY_GAMES,
                             num_simulations=MCTS_SIMS_SELFPLAY,
                             max_moves=MAX_GAME_MOVES,
                             temp_threshold=TEMP_THRESHOLD):
    """
    Chia n_games cho 2 GPU, chạy song song bằng threading.Thread.
    PyTorch release GIL khi gọi CUDA kernel → 2 GPU compute song song thực sự.
    CPU loop (96.6% là GPU time) không bị GIL block đáng kể.
    Fallback 1 GPU nếu chỉ có 1 GPU.
    """
    import threading
    half = n_games // 2

    if N_GPUS < 2:
        print(f"  [1 GPU fallback] {n_games} games trên {DEV0}")
        out = [None]
        _selfplay_on_device(model, DEV0, n_games,
                            num_simulations, max_moves, temp_threshold,
                            out, 0)
        return out[0]

    # threading thay mp.Process — Jupyter không hỗ trợ spawn
    # PyTorch release GIL khi gọi CUDA kernel → threading vẫn song song thực sự
    # cho phần chiếm 96.6% thời gian (GPU compute)
    out    = [None, None]
    model0 = model.to(DEV0); model0.eval()
    model1 = copy.deepcopy(model).to(DEV1); model1.eval()

    t0 = threading.Thread(
        target=_selfplay_on_device,
        args=(model0, DEV0, half,
              num_simulations, max_moves, temp_threshold, out, 0)
    )
    t1 = threading.Thread(
        target=_selfplay_on_device,
        args=(model1, DEV1, n_games - half,
              num_simulations, max_moves, temp_threshold, out, 1)
    )
    t0.start(); t1.start()
    t0.join();  t1.join()

    # Gộp kết quả
    states   = np.concatenate([out[0][0], out[1][0]])
    policies = np.concatenate([out[0][1], out[1][1]])
    wdl      = np.concatenate([out[0][2], out[1][2]])
    fens     = out[0][3] + out[1][3]

    valid    = policies.sum(axis=1) > 0
    states   = states[valid]
    policies = policies[valid]
    wdl      = wdl[valid]
    fens     = [f for f, v in zip(fens, valid) if v]

    print(f"  → {len(states):,} positions tổng (2 GPU Process)")
    return states, policies, wdl, fens


# ══════════════════════════════════════════════════════════
# ❽  DUAL-GPU EVALUATION  (BatchedMCTS + multiprocessing)
#
#  Thay MCTSEngine sequential bằng BatchedMCTS:
#  - n_games ván eval chạy song song trong cùng 1 batch
#  - Đến lượt phe nào đi → gom indices phe đó → step() engine tương ứng
#  - threading.Thread → 2 GPU compute song song (PyTorch release GIL)
# ══════════════════════════════════════════════════════════

def _eval_on_device(new_model, old_model, device, n_games, out_results):
    """
    Chạy n_games ván eval trên device bằng BatchedMCTS.
    Mỗi ván có opening ngẫu nhiên + new_is_white luân phiên.
    Kết quả ghi vào out_results dict.
    Model đã được đưa lên GPU bởi caller.
    """
    new_model = new_model.to(device)
    old_model = old_model.to(device)
    new_model.eval()
    old_model.eval()

    boards       = []
    last_moves   = []
    new_is_white = []
    for game_idx in range(n_games):
        b  = chess.Board()
        lm = _random_opening(b)
        boards.append(b)
        last_moves.append(lm)
        new_is_white.append(game_idx % 2 == 0)

    # boards_old: bản riêng cho eng_old track (push/pop độc lập)
    boards_old = [b.copy() for b in boards]
    done       = [False] * n_games
    new_wins   = draws = old_wins = 0

    eng_new = BatchedMCTS(new_model, device, n_games=n_games,
                          c_puct=C_PUCT,
                          dirichlet_alpha=0.3, dirichlet_eps=0.25)
    eng_old = BatchedMCTS(old_model, device, n_games=n_games,
                          c_puct=C_PUCT,
                          dirichlet_alpha=0.3, dirichlet_eps=0.25)

    eng_new.init_games(boards,      list(last_moves), add_noise=True)
    eng_old.init_games(boards_old,  list(last_moves), add_noise=True)

    while not all(done):
        active = [i for i in range(n_games) if not done[i]]
        if not active:
            break

        new_turn_idx = [i for i in active
                        if (boards[i].turn == chess.WHITE) == new_is_white[i]]
        old_turn_idx = [i for i in active
                        if (boards[i].turn == chess.WHITE) != new_is_white[i]]

        # Tạm mark inactive ván không đến lượt → step() bỏ qua
        saved_new = list(eng_new.active)
        saved_old = list(eng_old.active)
        for i in range(n_games):
            eng_new.active[i] = (i in new_turn_idx)
            eng_old.active[i] = (i in old_turn_idx)

        for _ in range(MCTS_SIMS_EVAL):
            if new_turn_idx: eng_new.step()
            if old_turn_idx: eng_old.step()

        for i in range(n_games):
            eng_new.active[i] = saved_new[i]
            eng_old.active[i] = saved_old[i]

        _, new_best = eng_new.get_results(boards)
        _, old_best = eng_old.get_results(boards_old)

        newly_done = []
        for i in active:
            is_new = (boards[i].turn == chess.WHITE) == new_is_white[i]
            chosen = new_best[i] if is_new else old_best[i]

            if chosen is None or chosen not in boards[i].legal_moves:
                legal  = list(boards[i].legal_moves)
                chosen = legal[0] if legal else None
            if chosen is None:
                done[i] = True
                newly_done.append(i)
                continue

            boards[i].push(chosen)
            boards_old[i].push(chosen)
            last_moves[i]         = chosen
            eng_new.last_moves[i] = chosen
            eng_old.last_moves[i] = chosen

            if boards[i].is_game_over():
                result = boards[i].result()
                niw    = new_is_white[i]
                if result == "1/2-1/2":
                    draws += 1
                elif (result == "1-0" and niw) or (result == "0-1" and not niw):
                    new_wins += 1
                else:
                    old_wins += 1
                done[i] = True
                newly_done.append(i)

        for i in newly_done:
            eng_new.active[i] = False; eng_new.roots[i] = None
            eng_old.active[i] = False; eng_old.roots[i] = None

        still_active = [i for i in active if not done[i]]
        if still_active:
            eng_new.partial_reinit(still_active, add_noise=True)
            eng_old.partial_reinit(still_active, add_noise=True)

    out_results["new_wins"] = out_results.get("new_wins", 0) + new_wins
    out_results["draws"]    = out_results.get("draws",    0) + draws
    out_results["old_wins"] = out_results.get("old_wins", 0) + old_wins


def evaluate_models_2gpu(new_model, old_model, device,
                          n_games=EVAL_GAMES,
                          num_simulations=MCTS_SIMS_EVAL):
    """
    Eval BatchedMCTS + threading cho 2 GPU.
    Mỗi GPU chạy n_games/2 ván song song.
    Fallback 1 GPU tự động.
    """
    import threading
    half = n_games // 2

    if N_GPUS < 2:
        results = {"new_wins": 0, "draws": 0, "old_wins": 0}
        _eval_on_device(new_model.to(device).eval(),
                        old_model.to(device).eval(),
                        device, n_games, results)
    else:
        res0 = {"new_wins": 0, "draws": 0, "old_wins": 0}
        res1 = {"new_wins": 0, "draws": 0, "old_wins": 0}
        lock = threading.Lock()   # protect res0/res1 nếu cần, nhưng thread riêng nên không cần

        new_m0 = new_model.to(DEV0); new_m0.eval()
        old_m0 = old_model.to(DEV0); old_m0.eval()
        new_m1 = copy.deepcopy(new_model).to(DEV1); new_m1.eval()
        old_m1 = copy.deepcopy(old_model).to(DEV1); old_m1.eval()

        t0 = threading.Thread(target=_eval_on_device,
                              args=(new_m0, old_m0, DEV0, half, res0))
        t1 = threading.Thread(target=_eval_on_device,
                              args=(new_m1, old_m1, DEV1, n_games - half, res1))
        t0.start(); t1.start()
        t0.join();  t1.join()

        results = {
            "new_wins": res0["new_wins"] + res1["new_wins"],
            "draws"   : res0["draws"]    + res1["draws"],
            "old_wins": res0["old_wins"] + res1["old_wins"],
        }

    nw       = results["new_wins"]
    d        = results["draws"]
    ow       = results["old_wins"]
    total    = nw + d + ow
    win_rate = (nw + 0.5 * d) / max(total, 1)
    print(f"  New {nw}W {d}D {ow}L | win_rate={win_rate:.3f}")
    return win_rate


# ══════════════════════════════════════════════════════════
# ❾  DATASET + DATALOADER
# ══════════════════════════════════════════════════════════

class SelfPlayDataset(torch.utils.data.Dataset):
    def __init__(self, states, policies, wdl, fens):
        self.states   = torch.from_numpy(states)
        self.policies = torch.from_numpy(policies)
        self.wdl      = torch.from_numpy(wdl)
        self.fens     = fens

    def __len__(self):
        return len(self.states)

    def __getitem__(self, idx):
        return (self.states[idx], self.policies[idx],
                self.wdl[idx], self.fens[idx])


def rl_collate(batch):
    states   = torch.stack([b[0] for b in batch])
    policies = torch.stack([b[1] for b in batch])
    wdl      = torch.stack([b[2] for b in batch])
    fens     = [b[3] for b in batch]
    return states, policies, wdl, fens


# ══════════════════════════════════════════════════════════
# ❿  RL TRAINING EPOCH
# ══════════════════════════════════════════════════════════

def rl_train_epoch(model, loader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = total_p = total_v = total = 0

    pbar = tqdm(loader, desc="  RL Train", leave=False, dynamic_ncols=True)
    for states, mcts_policies, wdl, fens in pbar:
        states        = states.to(device)
        mcts_policies = mcts_policies.to(device)
        wdl           = wdl.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            policy_logits, value_logits = model(states)
            masks       = batch_legal_masks(fens, device)
            log_probs   = F.log_softmax(policy_logits + masks, dim=1)
            policy_loss = -(mcts_policies * log_probs).sum(1).mean()
            value_loss  = F.cross_entropy(value_logits, wdl)
            loss        = policy_loss + value_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        n           = len(states)
        total_loss += loss.item() * n
        total_p    += policy_loss.item() * n
        total_v    += value_loss.item() * n
        total      += n
        pbar.set_postfix(p=f"{policy_loss.item():.3f}",
                         v=f"{value_loss.item():.3f}")

    return total_loss / total, total_p / total, total_v / total


# ══════════════════════════════════════════════════════════
# ⓫  BENCHMARK
# ══════════════════════════════════════════════════════════

def benchmark_v4(model, n_games=6, sims=100):
    import time
    print(f"\n{'='*50}")
    print(f"  Benchmark v4: {n_games} ván × {sims} sims ({N_GPUS} GPU)")
    print(f"{'='*50}")
    t0 = time.perf_counter()
    states, _, _, _ = generate_self_play_2gpu(
        model, n_games=n_games, num_simulations=sims)
    elapsed = time.perf_counter() - t0
    print(f"  {len(states):,} positions | {elapsed:.1f}s | "
          f"{len(states)/elapsed:.1f} pos/sec")
    print(f"  V2 baseline: ~{69.2/3*n_games:.0f}s → speedup ~{69.2/3*n_games/elapsed:.1f}x")


# ══════════════════════════════════════════════════════════
# ⓬  RL LOOP CHÍNH
# ══════════════════════════════════════════════════════════

def run_rl_loop(start_iteration=0):
    print("=" * 60)
    print(f"  RL LOOP v4  (Dual-GPU Batched MCTS, {N_GPUS} GPU detected)")
    print("=" * 60)

    best_model = ChessNet(
        channels=CHANNELS, res_blocks=RES_BLOCKS,
        attn_heads=ATTN_HEADS, attn_every=ATTN_EVERY
    ).to(DEV0)

    if os.path.exists(BEST_MODEL_CKPT):
        best_model.load_state_dict(
            torch.load(BEST_MODEL_CKPT, map_location=DEV0))
        print(f"✓ Loaded best model: {BEST_MODEL_CKPT}")
    else:
        best_model.load_state_dict(
            torch.load(PRETRAIN_CKPT, map_location=DEV0))
        torch.save(best_model.state_dict(), BEST_MODEL_CKPT)
        print(f"✓ Khởi tạo từ pretrain: {PRETRAIN_CKPT}")

    print("\n[Benchmark...]")
    benchmark_v4(best_model, n_games=4, sims=50)

    scaler        = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    replay_buffer = deque(maxlen=4)

    for iteration in range(start_iteration, RL_ITERATIONS):
        print(f"\n{'='*60}")
        print(f"  ITERATION {iteration+1}/{RL_ITERATIONS}")
        print(f"{'='*60}")

        # ── 1. Self-play (2 GPU song song) ────────────────
        print(f"\n[1/4] Self-play ({NUM_SELF_PLAY_GAMES} games, "
              f"{MCTS_SIMS_SELFPLAY} sims, {N_GPUS} GPU)...")
        states, policies, wdl, fens = generate_self_play_2gpu(
            best_model,
            n_games=NUM_SELF_PLAY_GAMES,
            num_simulations=MCTS_SIMS_SELFPLAY,
        )
        replay_buffer.append((states, policies, wdl, fens))

        # ── 2. Gom replay buffer + train ──────────────────
        buf_len      = len(replay_buffer)
        weight_steps = np.linspace(0.25, 1.0, buf_len)

        all_states, all_policies, all_wdl, all_fens = [], [], [], []
        all_weights = []
        for w, (s, p, v, f) in zip(weight_steps, replay_buffer):
            all_states.append(s)
            all_policies.append(p)
            all_wdl.append(v)
            all_fens.extend(f)
            all_weights.extend([w] * len(s))

        all_states   = np.concatenate(all_states)
        all_policies = np.concatenate(all_policies)
        all_wdl      = np.concatenate(all_wdl)
        sample_weights = torch.tensor(all_weights, dtype=torch.float32)

        n_pos = len(all_states)
        print(f"\n[2/4] Training ({n_pos:,} pos từ {buf_len} iter, "
              f"{RL_EPOCHS} epochs)...")

        new_model = copy.deepcopy(best_model).to(DEV0)
        optimizer = torch.optim.AdamW(new_model.parameters(),
                                      lr=RL_LR, weight_decay=RL_WEIGHT_DECAY)
        dataset   = SelfPlayDataset(all_states, all_policies, all_wdl, all_fens)
        sampler   = torch.utils.data.WeightedRandomSampler(
            weights=sample_weights, num_samples=len(dataset), replacement=True)
        loader    = torch.utils.data.DataLoader(
            dataset, batch_size=RL_BATCH_SIZE,
            sampler=sampler, num_workers=0,
            drop_last=True, collate_fn=rl_collate)

        rl_total_steps  = len(loader) * RL_EPOCHS
        rl_warmup_steps = max(1, rl_total_steps // 10)
        print(f"  → {rl_total_steps} steps | {rl_warmup_steps} warmup "
              f"| weights {weight_steps.round(2)}")

        rl_scheduler = make_scheduler(
            optimizer,
            warmup_steps=rl_warmup_steps,
            total_steps=rl_total_steps, eta_min=1e-6)

        for epoch in range(1, RL_EPOCHS + 1):
            loss, p_loss, v_loss = rl_train_epoch(
                new_model, loader, optimizer, rl_scheduler, scaler, DEV0)
            lr = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch}/{RL_EPOCHS}: "
                  f"Loss={loss:.4f} (P={p_loss:.4f} V={v_loss:.4f}) LR={lr:.2e}")

        # ── 3. Evaluate (2 GPU song song) ─────────────────
        print(f"\n[3/4] Evaluate ({EVAL_GAMES} games, {N_GPUS} GPU)...")
        new_model.eval(); best_model.eval()
        win_rate = evaluate_models_2gpu(
            new_model, best_model, DEV0, n_games=EVAL_GAMES)

        # ── 4. Replace ─────────────────────────────────────
        print(f"\n[4/4] win_rate={win_rate:.3f} | threshold={WIN_THRESHOLD}")
        if win_rate >= WIN_THRESHOLD:
            best_model = new_model.to(DEV0)
            torch.save(best_model.state_dict(), BEST_MODEL_CKPT)
            print(f"  ✅ Replace → {BEST_MODEL_CKPT}")
        else:
            print(f"  ❌ Giữ nguyên best model")

        ckpt = os.path.join(RL_CKPT_DIR, f"iter_{iteration+1:03d}.pt")
        torch.save({"iteration": iteration + 1,
                    "model"    : best_model.state_dict(),
                    "win_rate" : win_rate}, ckpt)
        print(f"  💾 {ckpt}")

    print(f"\n✅ RL xong! Best model: {BEST_MODEL_CKPT}")
    return best_model


# ══════════════════════════════════════════════════════════
# ⓭  VS STOCKFISH TEST
# ══════════════════════════════════════════════════════════

def play_vs_stockfish(model, stockfish_elo=1400,
                      n_games=10, num_simulations=200):
    engine_sf = chess.engine.SimpleEngine.popen_uci("stockfish")
    engine_sf.configure({"UCI_LimitStrength": True, "UCI_Elo": stockfish_elo})
    mcts    = MCTSEngine(model, DEV0, c_puct=C_PUCT)
    results = {"win": 0, "draw": 0, "loss": 0}

    for game_idx in range(n_games):
        board        = chess.Board()
        last_move    = None
        bot_is_white = (game_idx % 2 == 0)
        while not board.is_game_over():
            if (board.turn == chess.WHITE) == bot_is_white:
                move = mcts.best_move(board, last_move, num_simulations)
                if move is None or move not in board.legal_moves:
                    move = list(board.legal_moves)[0]
            else:
                result = engine_sf.play(board, chess.engine.Limit(time=0.1))
                move   = result.move
            board.push(move)
            last_move = move

        outcome = board.outcome()
        if outcome is None or outcome.winner is None:
            results["draw"] += 1
        elif (outcome.winner == chess.WHITE) == bot_is_white:
            results["win"] += 1
        else:
            results["loss"] += 1
        print(f"  Game {game_idx+1}: {'W' if bot_is_white else 'B'} → {board.result()}")

    engine_sf.quit()
    total    = sum(results.values())
    win_rate = (results["win"] + 0.5 * results["draw"]) / max(total, 1)
    print(f"\nVs Stockfish {stockfish_elo}: {results} | win_rate={win_rate:.3f}")
    return results


# ══════════════════════════════════════════════════════════
# ── Chạy ──────────────────────────────────────────────────
# benchmark_v4(model, DEVICE)
run_rl_loop(start_iteration=0)
# run_rl_loop(start_iteration=3)   # resume